# Project: BBLF AI Selector v2 
# Section: Model Feature Creation
## Sub Section: Pre Tournament

Goal: To clean the raw player data set and create required features for model build

Things to add:
1. Individual Player Actual Cricket Statistics (current season and previous seasons)
2. Power Surge (somehow/ backpopulate)
3. Adjust average fields (if not enough data should be null e.g. 3 game average should be null until atleast 3 games)
4. T stat/ Sharpe Ratio of current and lag points
5. Need metric/ feature to assess players joining late in the tournament (usual bad players)
6. Past batting and bowling average


# 0. Prerequistes

In [15]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import OneHotEncoder
from concurrent.futures import ProcessPoolExecutor

os.getcwd()
directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/'

In [16]:
# Type of response variable selection
response_var = "bowl_fp" # options: "bowl_fp", "bat_fp", "total_fp"

# 1. Data Extraction

In [17]:
# 1a. Pull in all_matches csv file 

data_prep_df = pd.read_csv(os.path.join(directory,'python_data/fant_point_play_tbl.csv'), low_memory=False)

# 1b. Group & fix mis match venue names
unique_venues = data_prep_df['venue'].unique()
print(unique_venues)

data_prep_df['venue'] = data_prep_df['venue'].replace('Western Australia Cricket Association Ground', 'WACA')
data_prep_df['venue'] = data_prep_df['venue'].replace('W.A.C.A. Ground', 'WACA')
data_prep_df['venue'] = data_prep_df['venue'].replace('Brisbane Cricket Ground, Woolloongabba', 'GABBA')
data_prep_df['venue'] = data_prep_df['venue'].replace('Brisbane Cricket Ground', 'GABBA')
data_prep_df['venue'] = data_prep_df['venue'].replace('Brisbane Cricket Ground, Woolloongabba, Brisbane', 'GABBA')
data_prep_df['venue'] = data_prep_df['venue'].replace('Aurora Stadium', 'Launceston')
data_prep_df['venue'] = data_prep_df['venue'].replace('University of Tasmania Stadium, Launceston', 'Launceston')
data_prep_df['venue'] = data_prep_df['venue'].replace('Aurora Stadium, Launceston', 'Launceston')
data_prep_df['venue'] = data_prep_df['venue'].replace('Manuka Oval', 'Manuka')
data_prep_df['venue'] = data_prep_df['venue'].replace('Manuka Oval, Canberra', 'Manuka')
data_prep_df['venue'] = data_prep_df['venue'].replace('Docklands Stadium, Melbourne', 'Marvel')
data_prep_df['venue'] = data_prep_df['venue'].replace('Docklands Stadium', 'Marvel')
data_prep_df['venue'] = data_prep_df['venue'].replace('International Sports Stadium, Coffs Harbour', 'Coffs Harbour')
data_prep_df['venue'] = data_prep_df['venue'].replace('International Sports Stadium', 'Coffs Harbour')
data_prep_df['venue'] = data_prep_df['venue'].replace('GMHBA Stadium, South Geelong, Victoria', 'Geelong')
data_prep_df['venue'] = data_prep_df['venue'].replace('Geelong Cricket Ground', 'Geelong')
data_prep_df['venue'] = data_prep_df['venue'].replace('Simonds Stadium, South Geelong, Victoria', 'Geelong')
data_prep_df['venue'] = data_prep_df['venue'].replace('Sydney Cricket Ground', 'SCG')
data_prep_df['venue'] = data_prep_df['venue'].replace('Lavington Sports Oval, Albury', 'Albury')
data_prep_df['venue'] = data_prep_df['venue'].replace('North Sydney Oval, Sydney', 'North Sydney Oval')
data_prep_df['venue'] = data_prep_df['venue'].replace("Cazaly's Stadium, Cairns", 'Cairns')
data_prep_df['venue'] = data_prep_df['venue'].replace("Junction Oval, Melbourne", 'Junction Oval')
data_prep_df['venue'] = data_prep_df['venue'].replace("Sydney Showground Stadium", 'Sydney Showground')
data_prep_df['venue'] = data_prep_df['venue'].replace("Stadium Australia", 'Sydney Showground')
data_prep_df['venue'] = data_prep_df['venue'].replace("Traeger Park", 'Alice Springs')
data_prep_df['venue'] = data_prep_df['venue'].replace("Bellerive Oval, Hobart", 'Hobart')
data_prep_df['venue'] = data_prep_df['venue'].replace("Bellerive Oval", 'Hobart')
data_prep_df['venue'] = data_prep_df['venue'].replace("Ted Summerton Reserve", 'Moe')
data_prep_df['venue'] = data_prep_df['venue'].replace("Carrara Oval", 'Gold Coast')

data_prep_df['venue'] = data_prep_df['venue'].replace('WACA', "Other")
data_prep_df['venue'] = data_prep_df['venue'].replace('Launceston', "Other")
data_prep_df['venue'] = data_prep_df['venue'].replace('Coffs Harbour', "Other")
data_prep_df['venue'] = data_prep_df['venue'].replace('Geelong', "Other")
data_prep_df['venue'] = data_prep_df['venue'].replace('Albury', "Other")
data_prep_df['venue'] = data_prep_df['venue'].replace('North Sydney Oval', "Other")
data_prep_df['venue'] = data_prep_df['venue'].replace('Cairns', "Other")
data_prep_df['venue'] = data_prep_df['venue'].replace('Junction Oval', "Other")
data_prep_df['venue'] = data_prep_df['venue'].replace('Alice Springs', "Other")
data_prep_df['venue'] = data_prep_df['venue'].replace('Moe', "Other")
data_prep_df['venue'] = data_prep_df['venue'].replace('Gold Coast', "Other")
data_prep_df['venue'] = data_prep_df['venue'].replace('Manuka', "Other")

unique_venues = data_prep_df['venue'].unique()
print(unique_venues)

unique_team = data_prep_df['team'].unique()
print(unique_team)

# 1c. Team Home and Away Flag
home_conditions = [
        (data_prep_df['venue'] == "Other"),
        (data_prep_df['venue'] == "GABBA") & (data_prep_df['team'] == "Brisbane Heat"),
        (data_prep_df['venue'] == "Melbourne Cricket Ground") & (data_prep_df['team'] == "Melbourne Stars"),
        (data_prep_df['venue'] == "Hobart") & (data_prep_df['team'] == "Hobart Hurricanes"),
        (data_prep_df['venue'] == "Sydney Showground") & (data_prep_df['team'] == "Sydney Thunder"),
        (data_prep_df['venue'] == "Adelaide Oval") & (data_prep_df['team'] == "Adelaide Strikers"),
        (data_prep_df['venue'] == "SCG") & (data_prep_df['team'] == "Sydney Sixers"),
        (data_prep_df['venue'] == "Marvel") & (data_prep_df['team'] == "Melbourne Renegades"),
        (data_prep_df['venue'] == "Perth Stadium") & (data_prep_df['team'] == "Perth Scorchers")
    ]

home_group = [0,1,1,1,1,1,1,1,1]

data_prep_df["Home_f"] = np.select(home_conditions, home_group)

['Western Australia Cricket Association Ground'
 'Brisbane Cricket Ground, Woolloongabba' 'Melbourne Cricket Ground'
 'Bellerive Oval' 'Sydney Showground Stadium' 'Adelaide Oval'
 'Sydney Cricket Ground' 'Docklands Stadium' 'Manuka Oval'
 'Stadium Australia' 'W.A.C.A. Ground'
 'Simonds Stadium, South Geelong, Victoria' 'Traeger Park'
 'Aurora Stadium' 'Perth Stadium' 'Carrara Oval' 'Geelong Cricket Ground'
 'Ted Summerton Reserve' 'International Sports Stadium'
 'Brisbane Cricket Ground' 'Manuka Oval, Canberra'
 'Docklands Stadium, Melbourne' 'Aurora Stadium, Launceston'
 'Bellerive Oval, Hobart'
 'Brisbane Cricket Ground, Woolloongabba, Brisbane'
 'Junction Oval, Melbourne' 'GMHBA Stadium, South Geelong, Victoria'
 'International Sports Stadium, Coffs Harbour' "Cazaly's Stadium, Cairns"
 'University of Tasmania Stadium, Launceston'
 'Lavington Sports Oval, Albury']
['Other' 'GABBA' 'Melbourne Cricket Ground' 'Hobart' 'Sydney Showground'
 'Adelaide Oval' 'SCG' 'Marvel' 'Perth Stadium']

# 1b. Imputing Missing Values

In [18]:
# a. Imputing missing batting position
# approach: Find the total number of runs scored by each player in the season and use this order to assign missing batting position (if tie, alphabetical order)

# 1. Identify player rank by total runs in season by team
player_season_runs = data_prep_df.groupby(['season', 'team', 'player'])['total_runs'].sum().reset_index()

# 2. Create loop to identify missing batting positions and assign based on total runs rank for each match and team
with ProcessPoolExecutor() as executor:
    for match_id in data_prep_df['match_id'].unique():
        # Get season and teams for the match
        season_id = data_prep_df.loc[data_prep_df['match_id'] == match_id, 'season'].values[0]
        teams = data_prep_df.loc[data_prep_df['match_id'] == match_id, 'team'].unique()
        
        for team in teams:
            # For each team, find players with missing batting positions
            team_players = data_prep_df[(data_prep_df['match_id'] == match_id) & (data_prep_df['team'] == team)]
            missing_batting_pos_players = team_players[team_players['bat_position'].isnull()]['player'].unique()
            
            # Assign batting positions based on total runs rank
            if len(missing_batting_pos_players) > 0:
                missing_batting_pos_cnt = len(missing_batting_pos_players)
                # Get player ranks for the team in the season
                # Get team player runs for the season
                team_player_runs = player_season_runs[(player_season_runs['season'] == season_id) & (player_season_runs['team'] == team)]
                # Filter to only players with missing batting positions in the match
                missing_player_runs = team_player_runs[team_player_runs['player'].isin(missing_batting_pos_players)]

                # Rank players by total runs (descending) and player name (ascending)
                missing_player_runs = missing_player_runs.sort_values(by=['total_runs', 'player'], ascending=[False, True]).reset_index(drop=True)
                missing_player_runs['rank'] = (11 - missing_batting_pos_cnt) + missing_player_runs.index + 1

                for player in missing_batting_pos_players:
                    # Get the player's rank
                    player_rank = missing_player_runs[missing_player_runs['player'] == player]['rank'].values[0]
                    data_prep_df.loc[(data_prep_df['match_id'] == match_id) & (data_prep_df['team'] == team) & (data_prep_df['player'] == player) & (data_prep_df['bat_position'].isnull()), 'bat_position'] = player_rank
        
            else:
                continue

# 3. Split batting position into binary features using one hot encoding
data_prep_df['bat_position'] = data_prep_df['bat_position'].astype(int)
data_prep_df = pd.get_dummies(data_prep_df, columns= ['bat_position'], prefix= 'bat_pos')
data_prep_df.loc[:,"bat_pos_1":"bat_pos_11"] = data_prep_df.loc[:,"bat_pos_1":"bat_pos_11"].astype(int)
data_prep_df.loc[:,"bat_pos_1":"bat_pos_11"] = data_prep_df.loc[:,"bat_pos_1":"bat_pos_11"].astype(object)

C:\Users\dilan\AppData\Local\Temp\ipykernel_11524\3391886389.py:43: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 0 0 ... 0 0 0]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  data_prep_df.loc[:,"bat_pos_1":"bat_pos_11"] = data_prep_df.loc[:,"bat_pos_1":"bat_pos_11"].astype(int)
C:\Users\dilan\AppData\Local\Temp\ipykernel_11524\3391886389.py:43: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 0 0 ... 0 0 0]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  data_prep_df.loc[:,"bat_pos_1":"bat_pos_11"] = data_prep_df.loc[:,"bat_pos_1":"bat_pos_11"].astype(int)
C:\Users\dilan\AppData\Local\Temp\ipykernel_11524\3391886389.py:43: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 0 0 ... 0 0 0]' h

In [19]:
# b. Imputing specific features missing values with 0
# Over bowled features
data_prep_df.loc[:,"over_1_f":"over_20_f"] = data_prep_df.loc[:,"over_1_f":"over_20_f"].fillna(0)

# Overs feature
data_prep_df['overs'] = data_prep_df['overs'].fillna(0)
 

# 2. Current Season Fantasy Point

In [20]:
if response_var == 'total_fp':
    season_df = data_prep_df.drop(['Unnamed: 0', 'team', 'opp', 'start_date', 'venue',
                                'first_bat_team', 'first_bowl_team', 'Home_f', 'match_id'], axis=1)
    season_df_agg = season_df.groupby(['player', 'season'], as_index=False).agg(
    # Exclude total points as different season had different number of matches
    # a. Fantasy Point Aggregations 
    season_fp =('fantasy_point',"sum"),
    avg_season_fp = ('fantasy_point', "mean"),
    max_season_fp = ('fantasy_point', 'max'),
    min_season_fp = ('fantasy_point', 'min'),
    sd_season_fp = ('fantasy_point', 'std'),
    match_cnt = ('fantasy_point', 'count'),

    # b. Fantasy Point Individual Batting Metrics Aggregations
    # Fantasy Total Bat Point Aggregations (null set to 0)
    # season_fp_bat =('fantasy_point_bat',"sum"),
    avg_season_fp_bat = ('fantasy_point_bat', "mean"),
    max_season_fp_bat = ('fantasy_point_bat', 'max'),
    min_season_fp_bat = ('fantasy_point_bat', 'min'),
    sd_season_fp_bat = ('fantasy_point_bat', 'std'),

    # Strike Rate Eligibility Aggregations (null set to 0)
    # season_fp_sre =('strike_rate_elig_f',"sum"),
    avg_season_fp_sre = ('strike_rate_elig_f', "mean"),

    # Strike Rate Group Aggregations (null set to 0)
    # season_fp_srg =('strike_rate_group',"sum"),
    avg_season_fp_srg = ('strike_rate_group', "mean"),
    max_season_fp_srg = ('strike_rate_group', 'max'),
    min_season_fp_srg = ('strike_rate_group', 'min'),
    sd_season_fp_srg = ('strike_rate_group', 'std'),

    # Run Bonus Group Aggregations (null set to 0)
    # season_fp_rbg =('run_bonus_group',"sum"),
    avg_season_fp_rbg = ('run_bonus_group', "mean"),

    # c. Fantasy Point Individual Bowling Metrics Aggregations
    # Fantasy Total Bowl Point Aggregations (null set to 0)
    # season_fp_bowl =('fantasy_point_bowl',"sum"),
    avg_season_fp_bowl = ('fantasy_point_bowl', "mean"),
    max_season_fp_bowl = ('fantasy_point_bowl', 'max'),
    min_season_fp_bowl = ('fantasy_point_bowl', 'min'),
    sd_season_fp_bowl = ('fantasy_point_bowl', 'std'),

    # Economy Rate Eligibility Aggregations (null set to 0)
    # season_fp_ere =('econ_elig_f',"sum"),
    avg_season_fp_ere = ('econ_elig_f', "mean"),

    # Economy Bonus Group Aggregations (null set to 0)
    # season_fp_ebg =('econ_bonus_group',"sum"),
    avg_season_fp_ebg = ('econ_bonus_group', "mean"),
    max_season_fp_ebg = ('econ_bonus_group', 'max'),
    min_season_fp_ebg = ('econ_bonus_group', 'min'),
    sd_season_fp_ebg = ('econ_bonus_group', 'std'),

    # Wicket Bonus Group Aggregations (null set to 0)
    # season_fp_wbg =('wicket_bonus_group',"sum"),
    avg_season_fp_wbg = ('wicket_bonus_group', "mean"),

    # d. Actual Batting Aggregations 
    # Total Runs
    # season_runs =('total_runs',"sum"),
    avg_season_runs = ('total_runs', "mean"),
    max_season_runs = ('total_runs', 'max'),
    min_season_runs = ('total_runs', 'min'),
    sd_season_runs = ('total_runs', 'std'),
    bat_cnt = ('total_runs', 'count'),

    # Strike Rate
    avg_season_strike = ('strike_rate', "mean"),
    max_season_strike = ('strike_rate', 'max'),
    min_season_strike = ('strike_rate', 'min'),
    sd_season_strike = ('strike_rate', 'std'),

    # e. Actual Bowling Aggregations
    # Season Over Bowled Flags
    # season_over_1_f = ('over_1_f', "sum"),
    # season_over_2_f = ('over_2_f', "sum"),
    # season_over_3_f = ('over_3_f', "sum"),
    # season_over_4_f = ('over_4_f', "sum"),
    # season_over_5_f = ('over_5_f', "sum"),
    # season_over_6_f = ('over_6_f', "sum"),
    # season_over_7_f = ('over_7_f', "sum"),
    # season_over_8_f = ('over_8_f', "sum"),
    # season_over_9_f = ('over_9_f', "sum"),
    # season_over_10_f = ('over_10_f', "sum"),
    # season_over_11_f = ('over_11_f', "sum"),
    # season_over_12_f = ('over_12_f', "sum"),
    # season_over_13_f = ('over_13_f', "sum"),
    # season_over_14_f = ('over_14_f', "sum"),
    # season_over_15_f = ('over_15_f', "sum"),
    # season_over_16_f = ('over_16_f', "sum"),
    # season_over_17_f = ('over_17_f', "sum"),
    # season_over_18_f = ('over_18_f', "sum"),
    # season_over_19_f = ('over_19_f', "sum"),
    # season_over_20_f = ('over_20_f', "sum"),

    # Distribution of Overs Bowled Flag
    avg_over_1_f = ('over_1_f', "mean"),
    avg_over_2_f = ('over_2_f', "mean"),
    avg_over_3_f = ('over_3_f', "mean"),
    avg_over_4_f = ('over_4_f', "mean"),
    avg_over_5_f = ('over_5_f', "mean"),
    avg_over_6_f = ('over_6_f', "mean"),
    avg_over_7_f = ('over_7_f', "mean"),
    avg_over_8_f = ('over_8_f', "mean"),
    avg_over_9_f = ('over_9_f', "mean"),
    avg_over_10_f = ('over_10_f', "mean"),
    avg_over_11_f = ('over_11_f', "mean"),
    avg_over_12_f = ('over_12_f', "mean"),
    avg_over_13_f = ('over_13_f', "mean"),
    avg_over_14_f = ('over_14_f', "mean"),
    avg_over_15_f = ('over_15_f', "mean"),
    avg_over_16_f = ('over_16_f', "mean"),
    avg_over_17_f = ('over_17_f', "mean"),
    avg_over_18_f = ('over_18_f', "mean"),
    avg_over_19_f = ('over_19_f', "mean"),
    avg_over_20_f = ('over_20_f', "mean"),

    # Over Bowled Flag
    over_1_f = ('over_1_f', "max"),
    over_2_f = ('over_2_f', "max"),
    over_3_f = ('over_3_f', "max"),
    over_4_f = ('over_4_f', "max"),
    over_5_f = ('over_5_f', "max"),
    over_6_f = ('over_6_f', "max"),
    over_7_f = ('over_7_f', "max"),
    over_8_f = ('over_8_f', "max"),
    over_9_f = ('over_9_f', "max"),
    over_10_f = ('over_10_f', "max"),
    over_11_f = ('over_11_f', "max"),
    over_12_f = ('over_12_f', "max"),
    over_13_f = ('over_13_f', "max"),
    over_14_f = ('over_14_f', "max"),
    over_15_f = ('over_15_f', "max"),
    over_16_f = ('over_16_f', "max"),
    over_17_f = ('over_17_f', "max"),
    over_18_f = ('over_18_f', "max"),
    over_19_f = ('over_19_f', "max"),
    over_20_f = ('over_20_f', "max"),

    # Number of Overs Bowled
    # season_overs =('overs',"sum"),
    avg_season_overs = ('overs', "mean"),
    max_season_overs = ('overs', 'max'),
    min_season_overs = ('overs', 'min'),
    sd_season_overs = ('overs', 'std'),
    bowl_cnt = ('overs', 'count'),

    # Wickets
    # season_wick =('wickets',"sum"),
    avg_season_wick = ('wickets', "mean"),
    max_season_wick = ('wickets', 'max'),
    min_season_wick = ('wickets', 'min'),
    sd_season_wick = ('wickets', 'std'),

    # Dot Balls
    # season_dots =('dot_ball_cnt',"sum"),
    avg_season_dots = ('dot_ball_cnt', "mean"),
    max_season_dots = ('dot_ball_cnt', 'max'),
    min_season_dots = ('dot_ball_cnt', 'min'),
    sd_season_dots = ('dot_ball_cnt', 'std'),

    # Extras
    # season_extras =('extra_cnt',"sum"),
    avg_season_extras = ('extra_cnt', "mean"),
    max_season_extras = ('extra_cnt', 'max'),
    min_season_extras = ('extra_cnt', 'min'),
    sd_season_extras = ('extra_cnt', 'std'),

    # Economy Rate
    avg_season_econ = ('extra_cnt', "mean"),
    max_season_econ = ('extra_cnt', 'max'),
    min_season_econ = ('extra_cnt', 'min'),
    sd_season_econ = ('extra_cnt', 'std'),
    )

In [21]:
if response_var == "bowl_fp":
    season_df = data_prep_df.drop(['Unnamed: 0', 'team', 'opp', 'start_date', 'venue',
                                'first_bat_team', 'first_bowl_team', 'Home_f', 'match_id'], axis=1)
    season_df_agg = season_df.groupby(['player', 'season'], as_index=False).agg(
    # Exclude total points as different season had different number of matches
    # a. Fantasy Point Individual Bowling Metrics Aggregations
    # Fantasy Total Bowl Point Aggregations (null set to 0)
    # season_fp_bowl =('fantasy_point_bowl',"sum"),
    avg_season_fp_bowl = ('fantasy_point_bowl', "mean"),
    max_season_fp_bowl = ('fantasy_point_bowl', 'max'),
    min_season_fp_bowl = ('fantasy_point_bowl', 'min'),
    sd_season_fp_bowl = ('fantasy_point_bowl', 'std'),

    # Economy Rate Eligibility Aggregations (null set to 0)
    # season_fp_ere =('econ_elig_f',"sum"),
    avg_season_fp_ere = ('econ_elig_f', "mean"),

    # Economy Bonus Group Aggregations (null set to 0)
    # season_fp_ebg =('econ_bonus_group',"sum"),
    avg_season_fp_ebg = ('econ_bonus_group', "mean"),
    max_season_fp_ebg = ('econ_bonus_group', 'max'),
    min_season_fp_ebg = ('econ_bonus_group', 'min'),
    sd_season_fp_ebg = ('econ_bonus_group', 'std'),

    # Wicket Bonus Group Aggregations (null set to 0)
    # season_fp_wbg =('wicket_bonus_group',"sum"),
    avg_season_fp_wbg = ('wicket_bonus_group', "mean"),

    # b. Actual Bowling Aggregations
    # # Season Over Bowled Flags
    # season_over_1_f = ('over_1_f', "sum"),
    # season_over_2_f = ('over_2_f', "sum"),
    # season_over_3_f = ('over_3_f', "sum"),
    # season_over_4_f = ('over_4_f', "sum"),
    # season_over_5_f = ('over_5_f', "sum"),
    # season_over_6_f = ('over_6_f', "sum"),
    # season_over_7_f = ('over_7_f', "sum"),
    # season_over_8_f = ('over_8_f', "sum"),
    # season_over_9_f = ('over_9_f', "sum"),
    # season_over_10_f = ('over_10_f', "sum"),
    # season_over_11_f = ('over_11_f', "sum"),
    # season_over_12_f = ('over_12_f', "sum"),
    # season_over_13_f = ('over_13_f', "sum"),
    # season_over_14_f = ('over_14_f', "sum"),
    # season_over_15_f = ('over_15_f', "sum"),
    # season_over_16_f = ('over_16_f', "sum"),
    # season_over_17_f = ('over_17_f', "sum"),
    # season_over_18_f = ('over_18_f', "sum"),
    # season_over_19_f = ('over_19_f', "sum"),
    # season_over_20_f = ('over_20_f', "sum"),

    # Distribution of Overs Bowled Flag
    avg_over_1_f = ('over_1_f', "mean"),
    avg_over_2_f = ('over_2_f', "mean"),
    avg_over_3_f = ('over_3_f', "mean"),
    avg_over_4_f = ('over_4_f', "mean"),
    avg_over_5_f = ('over_5_f', "mean"),
    avg_over_6_f = ('over_6_f', "mean"),
    avg_over_7_f = ('over_7_f', "mean"),
    avg_over_8_f = ('over_8_f', "mean"),
    avg_over_9_f = ('over_9_f', "mean"),
    avg_over_10_f = ('over_10_f', "mean"),
    avg_over_11_f = ('over_11_f', "mean"),
    avg_over_12_f = ('over_12_f', "mean"),
    avg_over_13_f = ('over_13_f', "mean"),
    avg_over_14_f = ('over_14_f', "mean"),
    avg_over_15_f = ('over_15_f', "mean"),
    avg_over_16_f = ('over_16_f', "mean"),
    avg_over_17_f = ('over_17_f', "mean"),
    avg_over_18_f = ('over_18_f', "mean"),
    avg_over_19_f = ('over_19_f', "mean"),
    avg_over_20_f = ('over_20_f', "mean"),

    # Over Bowled Flag
    over_1_f = ('over_1_f', "max"),
    over_2_f = ('over_2_f', "max"),
    over_3_f = ('over_3_f', "max"),
    over_4_f = ('over_4_f', "max"),
    over_5_f = ('over_5_f', "max"),
    over_6_f = ('over_6_f', "max"),
    over_7_f = ('over_7_f', "max"),
    over_8_f = ('over_8_f', "max"),
    over_9_f = ('over_9_f', "max"),
    over_10_f = ('over_10_f', "max"),
    over_11_f = ('over_11_f', "max"),
    over_12_f = ('over_12_f', "max"),
    over_13_f = ('over_13_f', "max"),
    over_14_f = ('over_14_f', "max"),
    over_15_f = ('over_15_f', "max"),
    over_16_f = ('over_16_f', "max"),
    over_17_f = ('over_17_f', "max"),
    over_18_f = ('over_18_f', "max"),
    over_19_f = ('over_19_f', "max"),
    over_20_f = ('over_20_f', "max"),

    # Number of Overs Bowled
    # season_overs =('overs',"sum"),
    avg_season_overs = ('overs', "mean"),
    max_season_overs = ('overs', 'max'),
    min_season_overs = ('overs', 'min'),
    sd_season_overs = ('overs', 'std'),
    bowl_cnt = ('overs', 'count'),

    # Wickets
    # season_wick =('wickets',"sum"),
    avg_season_wick = ('wickets', "mean"),
    max_season_wick = ('wickets', 'max'),
    min_season_wick = ('wickets', 'min'),
    sd_season_wick = ('wickets', 'std'),

    # Dot Balls
    # season_dots =('dot_ball_cnt',"sum"),
    avg_season_dots = ('dot_ball_cnt', "mean"),
    max_season_dots = ('dot_ball_cnt', 'max'),
    min_season_dots = ('dot_ball_cnt', 'min'),
    sd_season_dots = ('dot_ball_cnt', 'std'),

    # Extras
    # season_extras =('extra_cnt',"sum"),
    avg_season_extras = ('extra_cnt', "mean"),
    max_season_extras = ('extra_cnt', 'max'),
    min_season_extras = ('extra_cnt', 'min'),
    sd_season_extras = ('extra_cnt', 'std'),

    # Economy Rate
    avg_season_econ = ('extra_cnt', "mean"),
    max_season_econ = ('extra_cnt', 'max'),
    min_season_econ = ('extra_cnt', 'min'),
    sd_season_econ = ('extra_cnt', 'std'),
    )

In [22]:
if response_var == 'bat_fp':
    season_df = data_prep_df.drop(['Unnamed: 0', 'team', 'opp', 'start_date', 'venue',
                                'first_bat_team', 'first_bowl_team', 'Home_f', 'match_id'], axis=1)
    season_df_agg = season_df.groupby(['player', 'season'], as_index=False).agg(
    # Exclude total points as different season had different number of matches

    # a. Fantasy Point Individual Batting Metrics Aggregations
    # Fantasy Total Bat Point Aggregations (null set to 0)
    # season_fp_bat =('fantasy_point_bat',"sum"),
    avg_season_fp_bat = ('fantasy_point_bat', "mean"),
    max_season_fp_bat = ('fantasy_point_bat', 'max'),
    min_season_fp_bat = ('fantasy_point_bat', 'min'),
    sd_season_fp_bat = ('fantasy_point_bat', 'std'),

    # Strike Rate Eligibility Aggregations (null set to 0)
    # season_fp_sre =('strike_rate_elig_f',"sum"),
    avg_season_fp_sre = ('strike_rate_elig_f', "mean"),

    # Strike Rate Group Aggregations (null set to 0)
    # season_fp_srg =('strike_rate_group',"sum"),
    avg_season_fp_srg = ('strike_rate_group', "mean"),
    max_season_fp_srg = ('strike_rate_group', 'max'),
    min_season_fp_srg = ('strike_rate_group', 'min'),
    sd_season_fp_srg = ('strike_rate_group', 'std'),

    # Run Bonus Group Aggregations (null set to 0)
    # season_fp_rbg =('run_bonus_group',"sum"),
    avg_season_fp_rbg = ('run_bonus_group', "mean"),

    # b. Actual Batting Aggregations 
    # Total Runs
    # season_runs =('total_runs',"sum"),
    avg_season_runs = ('total_runs', "mean"),
    max_season_runs = ('total_runs', 'max'),
    min_season_runs = ('total_runs', 'min'),
    sd_season_runs = ('total_runs', 'std'),
    bat_cnt = ('total_runs', 'count'),

    # Strike Rate
    avg_season_strike = ('strike_rate', "mean"),
    max_season_strike = ('strike_rate', 'max'),
    min_season_strike = ('strike_rate', 'min'),
    sd_season_strike = ('strike_rate', 'std'),

    # Season Batting Position
    
    )

# 3. Response Variable
1. Individual Game Points
2. Player Season Standard Deviation (less complex for simulation tool)

In [23]:
# Create response variable dataframe based on specified response variable
if response_var == 'total_fp':
    resp_df = season_df_agg[["player", "season", "avg_season_fp", "match_cnt"]].rename(columns = {"avg_season_fp": "resp_var"})
elif response_var == 'bowl_fp':
    resp_df = season_df_agg[["player", "season", "avg_season_fp_bowl", "bowl_cnt"]].rename(columns = {"avg_season_fp_bowl": "resp_var_bowl"})
elif response_var == 'bat_fp':  
    resp_df = season_df_agg[["player", "season", "avg_season_fp_bat", "bat_cnt"]].rename(columns = {"avg_season_fp_bat": "resp_var_bat"})

else:
    raise ValueError("Invalid response variable specified. Choose from 'bowl_fp', 'bat_fp', or 'total_fp'.")

# resp_df = data_prep_df[["match_id", "player", "season", "fantasy_point", "team", "opp"]].rename(columns = {"fantasy_point": "resp_var"})


# 4. Feature Creation

In [24]:
# 4a. Previous Season/s Fantasy Point Summary Stats 
# Lag 1
season_df_lag1_stat = season_df_agg.copy()
season_df_lag1_stat['season'] = season_df_lag1_stat['season'] + 1
season_df_lag1_stat = season_df_lag1_stat.rename(columns={c:c+'_lag1' for c in season_df_lag1_stat.columns if c not in ['player', 'season']})

fant_model_df = pd.merge(resp_df, season_df_lag1_stat, left_on = ["player", "season"], right_on = ["player", "season"], how = "left")

# Lag 2
season_df_lag2_stat = season_df_agg.copy()
season_df_lag2_stat['season'] = season_df_lag2_stat['season'] + 2
season_df_lag2_stat = season_df_lag2_stat.rename(columns={c:c+'_lag2' for c in season_df_lag2_stat.columns if c not in ['player', 'season']})

fant_model_df = pd.merge(fant_model_df, season_df_lag2_stat, left_on = ["player", "season"], right_on = ["player", "season"], how = "left")

# Lag 3
season_df_lag3_stat = season_df_agg.copy()
season_df_lag3_stat['season'] = season_df_lag3_stat['season'] + 3
season_df_lag3_stat = season_df_lag3_stat.rename(columns={c:c+'_lag3' for c in season_df_lag3_stat.columns if c not in ['player', 'season']})

fant_model_df = pd.merge(fant_model_df , season_df_lag3_stat, left_on = ["player", "season"], right_on = ["player", "season"], how = "left")

# 4b. Individual Match Features (Not required for v2)
# Match Venue
# match_fact_df = data_prep_df[["match_id","player", "venue", "Home_f"]]
# fant_model_df = pd.merge(fant_model_df , match_fact_df, left_on = ["match_id", "player"], right_on = ["match_id", "player"], how = "left")

# # 4c. Opponent Season Power Ranking
# team_rank_df = pd.read_csv(os.path.join(directory,'add_data_created/pre_tourny/team_season_rank.csv'), low_memory=False).rename(columns = {"team": "opp"})
# fant_model_df = pd.merge(fant_model_df , team_rank_df, left_on = ["season", "opp"], right_on = ["season", "opp"], how = "left")

# 5. One-Hot Encoding Factor Features

In [25]:
# Create binary dummy variables
# fant_model_df = pd.get_dummies(fant_model_df, columns= ['opp','venue','rank_group'])

# Convert dummies to object type
# fant_model_df.loc[:,"opp_Adelaide Strikers":"opp_Sydney Thunder"] = fant_model_df.loc[:,"opp_Adelaide Strikers":"opp_Sydney Thunder"].astype(int)
# fant_model_df.loc[:,"opp_Adelaide Strikers":"opp_Sydney Thunder"] = fant_model_df.loc[:,"opp_Adelaide Strikers":"opp_Sydney Thunder"].astype(object)

# fant_model_df.loc[:,"venue_Adelaide Oval":"venue_Sydney Showground"] = fant_model_df.loc[:,"venue_Adelaide Oval":"venue_Sydney Showground"].astype(int)
# fant_model_df.loc[:,"venue_Adelaide Oval":"venue_Sydney Showground"] = fant_model_df.loc[:,"venue_Adelaide Oval":"venue_Sydney Showground"].astype(object)

# fant_model_df.loc[:,"rank_group_High":"rank_group_Middle"] = fant_model_df.loc[:,"rank_group_High":"rank_group_Middle"].astype(int)
# fant_model_df.loc[:,"rank_group_High":"rank_group_Middle"] = fant_model_df.loc[:,"rank_group_High":"rank_group_Middle"].astype(object)

# 6. Interaction Variables

In [26]:
# 6a. Venue & Home Flag (See if the certain teams have bigger home ground advantage)
# fant_model_df["Home_Adelaide Strikers"] = np.where((fant_model_df["venue_Adelaide Oval"] == 1) & (fant_model_df["Home_f"] == 1) , 1 , 0)
# fant_model_df["Home_Melbourne Stars"] = np.where((fant_model_df["venue_Melbourne Cricket Ground"] == 1) & (fant_model_df["Home_f"] == 1) , 1 , 0)
# fant_model_df["Home_Melbourne Renegades"] = np.where((fant_model_df["venue_Marvel"] == 1) & (fant_model_df["Home_f"] == 1) , 1 , 0)
# fant_model_df["Home_Brisbane Heat"] = np.where((fant_model_df["venue_GABBA"] == 1) & (fant_model_df["Home_f"] == 1) , 1 , 0)
# fant_model_df["Home_Perth Scorchers"] = np.where((fant_model_df["venue_Perth Stadium"] == 1) & (fant_model_df["Home_f"] == 1) , 1 , 0)
# fant_model_df["Home_Sydney Sixers"] = np.where((fant_model_df["venue_SCG"] == 1) & (fant_model_df["Home_f"] == 1) , 1 , 0)
# fant_model_df["Home_Sydney Thunder"] = np.where((fant_model_df["venue_Sydney Showground"] == 1) & (fant_model_df["Home_f"] == 1) , 1 , 0)
# fant_model_df["Home_Hobart Hurricanes"] = np.where((fant_model_df["venue_Hobart"] == 1) & (fant_model_df["Home_f"] == 1) , 1 , 0)

# fant_model_df.loc[:,"Home_Adelaide Strikers":"Home_Hobart Hurricanes"] = fant_model_df.loc[:,"Home_Adelaide Strikers":"Home_Hobart Hurricanes"].astype(object)

# 7. Final Modelling Dataset

In [27]:
# Different model dataset exclusions depending on response variable
if 'resp_var' in fant_model_df.columns:
    # Exclusion
    # 1. Remove first four season (due to lag variables up to 3 seasons prior. Also remove as over 10 years old) - currently 14 seasons of data
    print(f'Original rows: {len(fant_model_df)}')
    fant_model_df = fant_model_df[fant_model_df["season"] > 4]    
    print(f'Rows after first excl: {len(fant_model_df)}')

    # 2. Remove players with less than three games played in the current season
    fant_model_df = fant_model_df[fant_model_df["match_cnt"] > 2] 
    print(f'Rows after second excl: {len(fant_model_df)}')

    # 3. drop match count column
    fant_model_df = fant_model_df.drop(['match_cnt', 'match_cnt_lag1', 'match_cnt_lag2', 'match_cnt_lag3'], axis=1)

    # Save final model dataframe
    fant_model_df.to_csv(os.path.join(directory,'python_data/bblf_v2_model_data_v4.csv'))
    print("Saved bblf_v2_model_data_v2.csv")

elif 'resp_var_bat' in fant_model_df.columns:
    # Exclusion
    # 1. Remove first four season (due to lag variables up to 3 seasons prior. Also remove as over 10 years old) - currently 14 seasons of data
    print(f'Original rows: {len(fant_model_df)}')
    fant_model_df = fant_model_df[fant_model_df["season"] > 4]    
    print(f'Rows after first excl: {len(fant_model_df)}')

    # 2. Remove players with less than three batting/ bowling games played in the current season
    fant_model_df = fant_model_df[fant_model_df["bat_cnt"] > 2]
    print(f'Rows after second excl: {len(fant_model_df)}')

    # 3. drop match count column
    fant_model_df = fant_model_df.drop(['bat_cnt', 'bat_cnt_lag1', 'bat_cnt_lag2', 'bat_cnt_lag3'], axis=1)

    # Save final model dataframe
    fant_model_df.to_csv(os.path.join(directory,'python_data/bblf_v2_model_data_v4_bat.csv'))
    print("Saved bblf_v2_model_data_v3_bat.csv")

elif 'resp_var_bowl' in fant_model_df.columns:
    # Exclusion
    # 1. Remove first four season (due to lag variables up to 3 seasons prior. Also remove as over 10 years old) - currently 14 seasons of data
    print(f'Original rows: {len(fant_model_df)}')
    fant_model_df = fant_model_df[fant_model_df["season"] > 4]    
    print(f'Rows after first excl: {len(fant_model_df)}')

    # 2. Remove players with less than three batting/ bowling games played in the current season
    fant_model_df = fant_model_df[fant_model_df["bowl_cnt"] > 2]
    print(f'Rows after second excl: {len(fant_model_df)}')

    # 3. drop match count column
    fant_model_df = fant_model_df.drop(['bowl_cnt', 'bowl_cnt_lag1', 'bowl_cnt_lag2', 'bowl_cnt_lag3'], axis=1)

    # Save final model dataframe
    fant_model_df.to_csv(os.path.join(directory,'python_data/bblf_v2_model_data_v4_bowl.csv'))
    print("Saved bblf_v2_model_data_v3_bowl.csv")

else:
    print("No valid response variable found in fant_model_df, uncommented the code in section 3.")

Original rows: 1990
Rows after first excl: 1456
Rows after second excl: 1229
Saved bblf_v2_model_data_v3_bowl.csv


# 8. Create BBL15 Previous Season Data Features for Model Scoring

In [28]:
# 8a. Lag Variables (incl latest BBL data for players who have played BBL but the previous season)
# #season_df_lag1_stat_14 = season_df_lag1_stat[season_df_lag1_stat["season"] == 14]
# #season_df_lag2_stat_14 = season_df_lag2_stat[season_df_lag2_stat["season"] == 14]
# #season_df_lag3_stat_14 = season_df_lag3_stat[season_df_lag3_stat["season"] == 14]
# #
# #season_df_lag_14 = pd.merge(season_df_lag1_stat_14 , season_df_lag2_stat_14, left_on = ["player","season"], right_on = ["player","season"], how = "left")
# #season_df_lag_14 = pd.merge(season_df_lag_14 , season_df_lag3_stat_14, left_on = ["player","season"], right_on = ["player","season"], how = "left")

# # player_df = pd.read_csv(os.path.join(directory,'data/python_datasets/player_price.csv'), low_memory=False)
# # player_df = player_df[["player", "Full_Name"]]

#     # Latest BBL player records
#  # 1. For loop for each distinct player to find max season
 
# player_list = season_df_lag1_stat.player.unique()
# #ind_play_lag_df = season_df_lag1_stat[season_df_lag1_stat["player"] == "JW Wells"]
# #ind_play_last_season = ind_play_lag_df[["season"]].max()
# play_last_season = []

# for x in player_list:

#     # select individual player records 
#     ind_play_lag_df = season_df_lag1_stat[season_df_lag1_stat["player"] == x]
#     ind_play_last_season = ind_play_lag_df[["season"]].sort_values(by='season', ascending=False).iloc[0]
#     ind_play_last_season = max(ind_play_last_season)
#     play_last_season = play_last_season + [ind_play_last_season]

# join_list = {'player': player_list, 'season': play_last_season}
# print(join_list)

# bbl14_play_lags = pd.DataFrame(join_list)

# all_play_latest_season_lags = pd.merge(bbl14_play_lags , season_df_lag1_stat, left_on = ["player","season"], right_on = ["player","season"], how = "left")
# all_play_latest_season_lags = pd.merge(all_play_latest_season_lags , season_df_lag2_stat, left_on = ["player","season"], right_on = ["player","season"], how = "left")
# all_play_latest_season_lags = pd.merge(all_play_latest_season_lags , season_df_lag3_stat, left_on = ["player","season"], right_on = ["player","season"], how = "left")
# all_play_latest_season_lags = all_play_latest_season_lags.drop(["season"], axis = 1)
# print(all_play_latest_season_lags)

# # Join all_player_latest_season_lags to the bbl14 player list
# season_df_lag_14 = pd.merge(player_df, all_play_latest_season_lags, left_on = ["player"], right_on = ["player"], how = "left")


# season_df_lag_14.to_csv('data/python_datasets/test.csv')

# season_df_lag_14.to_csv('data/python_datasets/bbl14_lags.csv')